## 06 — FotMob Bronze → Silver: matches, xG, shots, player stats


Unpacks `bronze.fotmob_matches_raw` into 4 silver tables. The key output is the shots table (`silver.fotmob_shots`) — it contains position, xG, and xGoT for every shot and will form the basis of the Gold layer build.

```
bronze.fotmob_matches_raw
    ├── silver.fotmob_matches          — match metadata (1 row per match)
    ├── silver.fotmob_team_xg          — match-level xG per team
    ├── silver.fotmob_shots            — individual shot events
    └── silver.fotmob_player_match     — aggregated player stats per match
```


### 1. Load Bronze / Wczytanie Bronze

**PL:** Wczytujemy tabelę bronze FotMob. Wypisujemy liczbę meczów jako szybką weryfikację.  
**EN:** Reads the FotMob bronze table. Prints the match count as a quick sanity-check.


In [ ]:
from pyspark.sql import functions as F

FOTMOB_BRONZE = "wsl_analytics.bronze.fotmob_matches_raw"

fotmob_bronze_df = spark.table(FOTMOB_BRONZE)

print("Liczba meczów:", fotmob_bronze_df.count())

display(fotmob_bronze_df)

### 2. silver.fotmob_matches — match metadata / Metadane meczów

**PL:** Tworzymy tabelę z metadanymi meczów: identyfikatory, datę, drużyny, wynik i poziom pokrycia danych (`coverage_level`). Pole `coverage_level` z FotMob wskazuje, jak kompletne są dane dla danego meczu (np. brak mapy strzałów dla niektórych meczów z niskim pokryciem).  
**EN:** Creates the match metadata table: identifiers, date, teams, score, and data coverage level (`coverage_level`). The `coverage_level` field from FotMob indicates how complete the data is for a given match (e.g. shot map may be absent for low-coverage matches).


In [ ]:
fotmob_matches_df = (
    fotmob_bronze_df
    .select(
        "fotmob_match_id",
        "league_id",
        "match_datetime",

        F.to_date("match_datetime")
            .alias("match_date"),

        "home_team_id",
        "away_team_id",

        F.try_variant_get(
            "payload",
            "$.general.homeTeam.name",
            "string"
        ).alias("home_team"),

        F.try_variant_get(
            "payload",
            "$.general.awayTeam.name",
            "string"
        ).alias("away_team"),

        F.try_variant_get(
            "payload",
            "$.general.matchName",
            "string"
        ).alias("match_name"),

        F.try_variant_get(
            "payload",
            "$.general.matchRound",
            "string"
        ).alias("round"),

        F.try_variant_get(
            "payload",
            "$.header.status.scoreStr",
            "string"
        ).alias("score"),

        "coverage_level",

        "source_file_name",
        "ingested_at"
    )
)

display(fotmob_matches_df)

### 4. Verify the number matches

In [ ]:
print(
    "Silver matches:",
    fotmob_matches_df.count()
)

### 4. Duplicate check 

Verifies no duplicates on `fotmob_match_id`.


In [ ]:
display(
    fotmob_matches_df
    .groupBy("fotmob_match_id")
    .count()
    .filter(F.col("count") > 1)
)

### 5. Write silver.fotmob_matches 

Writes the table.


In [ ]:
(
    fotmob_matches_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.fotmob_matches"
    )
)

### 6. silver.fotmob_team_xg — match-level xG 

Extracts match-level xG from the nested `content.stats.Periods.All.stats` structure in the FotMob JSON. Uses `LATERAL variant_explode()` in SQL to iterate over nested stat category arrays and select the one with key `expected_goals`. The result is one home/away xG pair per match.


In [ ]:
team_xg_wide_df = spark.sql("""
WITH categories AS (

    SELECT
        b.fotmob_match_id,
        b.home_team_id,
        b.away_team_id,
        category.value AS category

    FROM wsl_analytics.bronze.fotmob_matches_raw AS b,
    LATERAL variant_explode(
        b.payload:content.stats.Periods.All.stats
    ) AS category
),

stats AS (

    SELECT
        fotmob_match_id,
        home_team_id,
        away_team_id,
        stat.value AS stat

    FROM categories,
    LATERAL variant_explode(
        category:stats
    ) AS stat

    WHERE
        try_variant_get(
            category,
            '$.key',
            'string'
        ) = 'top_stats'
)

SELECT
    fotmob_match_id,
    home_team_id,
    away_team_id,

    try_variant_get(
        stat,
        '$.stats[0]',
        'double'
    ) AS home_xg,

    try_variant_get(
        stat,
        '$.stats[1]',
        'double'
    ) AS away_xg

FROM stats

WHERE
    try_variant_get(
        stat,
        '$.key',
        'string'
    ) = 'expected_goals'
""")

display(team_xg_wide_df)

In [ ]:
home_xg_df = (
    team_xg_wide_df
    .join(
        fotmob_matches_df.select(
            "fotmob_match_id",
            "home_team"
        ),
        on="fotmob_match_id"
    )
    .select(
        "fotmob_match_id",

        F.col("home_team_id")
            .alias("fotmob_team_id"),

        F.col("home_team")
            .alias("team_name"),

        F.lit("home")
            .alias("location"),

        F.col("home_xg")
            .alias("xg")
    )
)

In [ ]:
away_xg_df = (
    team_xg_wide_df
    .join(
        fotmob_matches_df.select(
            "fotmob_match_id",
            "away_team"
        ),
        on="fotmob_match_id"
    )
    .select(
        "fotmob_match_id",

        F.col("away_team_id")
            .alias("fotmob_team_id"),

        F.col("away_team")
            .alias("team_name"),

        F.lit("away")
            .alias("location"),

        F.col("away_xg")
            .alias("xg")
    )
)

### 7. Pivot to long format 

Pivots the wide DataFrame (home_xg + away_xg in one row) to long format (one row per team per match). Two separate joins with `fotmob_matches_df` add team names for home and away, which are then combined via `unionByName`.


In [ ]:
fotmob_team_xg_df = (
    home_xg_df
    .unionByName(
        away_xg_df
    )
)

display(fotmob_team_xg_df)

### 8. Validate xG pairs 

Each match should have exactly 2 rows (home + away). Matches with != 2 rows suggest missing xG data in source.


In [ ]:
display(
    fotmob_team_xg_df
    .groupBy("fotmob_match_id")
    .count()
    .filter(F.col("count") != 2)
)

### 9. Write silver.fotmob_team_xg 

Write.


In [ ]:
(
    fotmob_team_xg_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.fotmob_team_xg"
    )
)

### 10. silver.fotmob_shots — shot events 

Extracts all shots from the `content.shotmap.shots` array in each match. For each shot records:
- identifiers (match, team, player)
- minute and added time
- event type (`Miss`, `AttemptSaved`, `Goal`, etc.)
- shot position (x, y on pitch)
- quality metrics (xG, xGoT)
- flags (`is_on_target`, `is_blocked`, `is_own_goal`, `is_from_inside_box`)


In [ ]:
fotmob_shots_df = spark.sql("""
SELECT

    b.fotmob_match_id,

    try_variant_get(
        shot.value,
        '$.id',
        'bigint'
    ) AS shot_id,

    try_variant_get(
        shot.value,
        '$.teamId',
        'bigint'
    ) AS fotmob_team_id,

    try_variant_get(
        shot.value,
        '$.playerId',
        'bigint'
    ) AS fotmob_player_id,

    try_variant_get(
        shot.value,
        '$.playerName',
        'string'
    ) AS player_name,

    try_variant_get(
        shot.value,
        '$.min',
        'int'
    ) AS minute,

    try_variant_get(
        shot.value,
        '$.minAdded',
        'int'
    ) AS minute_added,

    try_variant_get(
        shot.value,
        '$.eventType',
        'string'
    ) AS event_type,

    try_variant_get(
        shot.value,
        '$.x',
        'double'
    ) AS x,

    try_variant_get(
        shot.value,
        '$.y',
        'double'
    ) AS y,

    try_variant_get(
        shot.value,
        '$.expectedGoals',
        'double'
    ) AS xg,

    try_variant_get(
        shot.value,
        '$.expectedGoalsOnTarget',
        'double'
    ) AS xgot,

    try_variant_get(
        shot.value,
        '$.shotType',
        'string'
    ) AS shot_type,

    try_variant_get(
        shot.value,
        '$.situation',
        'string'
    ) AS situation,

    try_variant_get(
        shot.value,
        '$.period',
        'string'
    ) AS period,

    try_variant_get(
        shot.value,
        '$.isOnTarget',
        'boolean'
    ) AS is_on_target,

    try_variant_get(
        shot.value,
        '$.isBlocked',
        'boolean'
    ) AS is_blocked,

    try_variant_get(
        shot.value,
        '$.isOwnGoal',
        'boolean'
    ) AS is_own_goal,

    try_variant_get(
        shot.value,
        '$.isFromInsideBox',
        'boolean'
    ) AS is_from_inside_box,

    try_variant_get(
        shot.value,
        '$.keeperId',
        'bigint'
    ) AS keeper_id

FROM
    wsl_analytics.bronze.fotmob_matches_raw AS b,

    LATERAL variant_explode(
        b.payload:content.shotmap.shots
    ) AS shot
""")

### 11. Preview shots 

Displays the shots DataFrame.


In [ ]:
display(fotmob_shots_df)

### 12. Add team name 

Joins with `fotmob_matches_df` to add team names (home/away) based on `fotmob_team_id`. The `CASE WHEN fotmob_team_id == home_team_id` condition determines whether a shot belongs to the home or away team.


In [ ]:
fotmob_shots_df = (
    fotmob_shots_df
    .join(
        fotmob_matches_df.select(
            "fotmob_match_id",
            "home_team_id",
            "home_team",
            "away_team_id",
            "away_team"
        ),
        on="fotmob_match_id",
        how="left"
    )
    .withColumn(
        "team_name",
        F.when(
            F.col("fotmob_team_id")
            == F.col("home_team_id"),
            F.col("home_team")
        )
        .when(
            F.col("fotmob_team_id")
            == F.col("away_team_id"),
            F.col("away_team")
        )
    )
    .drop(
        "home_team_id",
        "away_team_id",
        "home_team",
        "away_team"
    )
)

### 13. Preview with team names 

Displays shots with team names assigned.


In [ ]:
display(fotmob_shots_df)

### 14. Shot duplicate check 

Verifies no duplicates on (`fotmob_match_id`, `shot_id`).


In [ ]:
shot_duplicates_df = (
    fotmob_shots_df
    .groupBy(
        "fotmob_match_id",
        "shot_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

display(shot_duplicates_df)

### 15. Derive is_goal 

Creates `is_goal` by comparing `event_type == "Goal"`. Note: a bug existed in Silver where `is_goal` was NULL for all rows despite correct `event_type`. The fix is explicitly documented here. (Full repair is applied in notebook 08.)


In [ ]:
fotmob_shots_df = (
    fotmob_shots_df
    .withColumn(
        "is_goal",
        F.col("event_type") == "Goal"
    )
)

### 16. Preview selected fields 

Preview of key shot columns.


In [ ]:
display(
    fotmob_shots_df.select(
        "fotmob_match_id",
        "team_name",
        "player_name",
        "minute",
        "event_type",
        "xg",
        "xgot",
        "shot_type",
        "situation"
    )
)

### 17. Write silver.fotmob_shots 

Writes the shots table.


In [ ]:
(
    fotmob_shots_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.fotmob_shots"
    )
)

### 18. xG validation — shot sum vs match total 

Sums individual shot xG per team per match and compares with `silver.fotmob_team_xg` (match-level xG provided by FotMob). Differences arise from rounding — we expect values close to 0, not exactly 0.


In [ ]:
shot_xg_sum_df = (
    fotmob_shots_df
    .groupBy(
        "fotmob_match_id",
        "fotmob_team_id"
    )
    .agg(
        F.sum("xg")
        .alias("shot_xg_sum"),

        F.count("*")
        .alias("shots")
    )
)

In [ ]:
xg_validation_df = (
    fotmob_team_xg_df
    .join(
        shot_xg_sum_df,
        on=[
            "fotmob_match_id",
            "fotmob_team_id"
        ],
        how="left"
    )
    .withColumn(
        "xg_difference",

        F.round(
            F.col("xg")
            - F.col("shot_xg_sum"),
            4
        )
    )
)

In [ ]:
display(xg_validation_df)

### 19. silver.fotmob_player_stats_long — raw player stats 

Extracts player statistics from `content.playerStats` — a multi-level VARIANT structure. Uses `LATERAL variant_explode` three times: over players, stat categories, and individual metrics. The result is long-format data with one row per metric per player per match.


In [ ]:
fotmob_player_stats_long_df = spark.sql("""
WITH players AS (

    SELECT
        b.fotmob_match_id,
        player.key AS player_key,
        player.value AS player

    FROM
        wsl_analytics.bronze.fotmob_matches_raw AS b,

        LATERAL variant_explode(
            b.payload:content.playerStats
        ) AS player
),

categories AS (

    SELECT
        fotmob_match_id,
        player_key,
        player,

        category.value AS category

    FROM players,

    LATERAL variant_explode(
        player:stats
    ) AS category
),

metrics AS (

    SELECT
        fotmob_match_id,
        player_key,
        player,

        try_variant_get(
            category,
            '$.key',
            'string'
        ) AS stat_group,

        metric.key AS stat_name,

        metric.value AS metric

    FROM categories,

    LATERAL variant_explode(
        category:stats
    ) AS metric
)

SELECT

    fotmob_match_id,

    try_variant_get(
        player,
        '$.id',
        'bigint'
    ) AS fotmob_player_id,

    try_variant_get(
        player,
        '$.name',
        'string'
    ) AS player_name,

    try_variant_get(
        player,
        '$.teamId',
        'bigint'
    ) AS fotmob_team_id,

    try_variant_get(
        player,
        '$.teamName',
        'string'
    ) AS team_name,

    stat_group,
    stat_name,

    try_variant_get(
        metric,
        '$.key',
        'string'
    ) AS stat_key,

    try_variant_get(
        metric,
        '$.stat.value',
        'double'
    ) AS stat_value,

    try_variant_get(
        metric,
        '$.stat.total',
        'double'
    ) AS stat_total,

    try_variant_get(
        metric,
        '$.stat.type',
        'string'
    ) AS stat_type

FROM metrics
""")

In [ ]:
display(fotmob_player_stats_long_df)

In [ ]:
(
    fotmob_player_stats_long_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.fotmob_player_stats_long"
    )
)

### 20. silver.fotmob_player_match — aggregated player stats 

Pivots from long to wide format: one row per player per match. For each metric uses `MAX(WHEN stat_key == 'xyz' THEN stat_value)` as a conditional pivot — safe because each metric key appears at most once per player per match.


In [ ]:
fotmob_player_match_df = (
    fotmob_player_stats_long_df

    .groupBy(
        "fotmob_match_id",
        "fotmob_player_id",
        "player_name",
        "fotmob_team_id",
        "team_name"
    )

    .agg(

        F.max(
            F.when(
                F.col("stat_key") == "rating_title",
                F.col("stat_value")
            )
        ).alias("rating"),

        F.max(
            F.when(
                F.col("stat_key") == "minutes_played",
                F.col("stat_value")
            )
        ).alias("minutes_played"),

        F.max(
            F.when(
                F.col("stat_key") == "goals",
                F.col("stat_value")
            )
        ).alias("goals"),

        F.max(
            F.when(
                F.col("stat_key") == "assists",
                F.col("stat_value")
            )
        ).alias("assists"),

        F.max(
            F.when(
                F.col("stat_key") == "expected_goals",
                F.col("stat_value")
            )
        ).alias("xg"),

        F.max(
            F.when(
                F.col("stat_key")
                == "expected_goals_on_target_variant",
                F.col("stat_value")
            )
        ).alias("xgot"),

        F.max(
            F.when(
                F.col("stat_key") == "expected_assists",
                F.col("stat_value")
            )
        ).alias("xa"),

        F.max(
            F.when(
                F.col("stat_key") == "xg_and_xa",
                F.col("stat_value")
            )
        ).alias("xg_xa"),

        F.max(
            F.when(
                F.col("stat_key") == "total_shots",
                F.col("stat_value")
            )
        ).alias("total_shots"),

        F.max(
            F.when(
                F.col("stat_key")
                == "expected_goals_non_penalty",
                F.col("stat_value")
            )
        ).alias("npxg")
    )
)

In [ ]:
display(fotmob_player_match_df)

In [ ]:
(
    fotmob_player_match_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.fotmob_player_match"
    )
)

### 21. Show all silver tables 

Lists all silver tables — completeness verification.


In [ ]:
%sql

SHOW TABLES
IN wsl_analytics.silver;